# Data Loading

In [119]:
%%time

import kplex_dev
import data_splits
import warnings
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split



warnings.filterwarnings('ignore')

timeout_replacement = 1800  # max timeout replacement
excel_path = "kplex_ML_data.xlsx"

feature_cols = data_splits.feature_cols
algorithms = data_splits.algorithms


CPU times: total: 15.6 ms
Wall time: 15.6 ms


In [ ]:
# Loading and preprocessing data
cat_df, dfs = kplex_dev.load_excel_data(excel_path, timeout_replacement)
kplex_dev.process_kcore_list_to_statistics(cat_df)

# Selecting weight scheme
epsilon = 0.001 # need to set !!!
scheme = kplex_dev.W_TIME_MIN_X_W_LOG_TIME_RATIO # Change this and analyse!!

# Assigning weights
kplex_dev.assign_weights(cat_df, scheme, epsilon)

# kplex_dev.export_sorted(cat_df)

# Data reformatting
cat_df = kplex_dev.post_process_data(cat_df)

# fastest_time = minimum across those algorithms
cat_df['fastest_time'] = cat_df[data_splits.algorithms].min(axis=1)
cat_df

,dataset,k_value,n,m,d_avg,d_max,core_max,core_min,shell_num,shell_avgSize,shell_maxSize,shell_minSize,shell_stdSize,weight,UMKP,AltRB,kPEX,Pivot_Br,S_Br,fastest_time
0,MANN-a81,20,3321,5506380,1658.0488,3317,3280,3280,1,3321.000000,3321,3321,0.000000,306.046574,1459.5,1800.0,1468.1,1477.4,1504.7,1459.5
1,wikipedia_link_sr,20,1810,1637059,904.4525,1809,1798,1795,4,1806.250000,1810,1803,2.861381,66.902739,22.6,18.0,737.7,740.4,738.1,18.0
2,scc_reality,20,1332,880733,661.2110,1331,1235,1230,6,1290.333333,1332,1236,35.714921,47.684856,1800.0,1214.8,9.4,9.7,9.0,9.0
3,web-wikipedia_link_en13-all,20,432,93085,215.4745,431,427,427,1,432.000000,432,432,0.000000,184.662528,239.3,287.1,517.7,504.4,516.7,239.3
4,livejournal-links,20,442,94435,213.6538,441,371,369,3,412.333333,442,386,22.983086,0.269931,1.3,1.6,1.6,1.6,1.6,1.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
755,soc-flixster,2,264,11792,44.6667,150,67,52,16,215.000000,264,125,56.656862,0.784308,2.0,0.6,0.5,0.5,2.4,0.5
756,edit-ukwiki,2,281,14183,50.4733,280,65,40,26,235.384615,281,168,34.703222,0.437734,1.2,1.1,0.6,0.6,0.5,0.5
757,johnson8-4-4,2,70,1855,26.5000,53,53,53,1,70.000000,70,70,0.000000,0.389785,0.8,0.9,1.1,1.1,0.3,0.3
758,hamming6-2,2,64,1824,28.5000,57,57,57,1,64.000000,64,64,0.000000,0.081093,0.2,0.3,0.2,0.2,0.3,0.2


# Training with Neural Network

Simple neural network with a soft‑cross‑entropy loss.


### Model

In [21]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import TensorDataset, DataLoader
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.preprocessing import StandardScaler
from sklearn.utils.validation import check_array, check_is_fitted


class SoftCrossEntropyLoss(nn.Module):
    def __init__(self, temperature: float = 1.0):
        super().__init__()
        self.temperature = temperature

    def forward(self, logits: torch.Tensor, targets: torch.Tensor):
        scaled = logits / self.temperature
        log_probs = F.log_softmax(scaled, dim=1)
        # print('log_probs', log_probs, 'targets', targets)
        return -torch.sum(targets * log_probs, dim=1).mean()

# class SoftCrossEntropyLoss(nn.Module):
#     def __init__(self, temperature: float = 1.0, clamp_min: float = -20.0):
#         super().__init__()
#         self.temperature = temperature
#         self.clamp_min   = clamp_min

#     def forward(self, logits: torch.Tensor, targets: torch.Tensor):
#         # scale & log‑softmax
#         scaled    = logits / self.temperature
#         log_probs = F.log_softmax(scaled, dim=1)
#         # clamp to avoid -Inf
#         log_probs = torch.clamp(log_probs, min=self.clamp_min)
#         # ensure targets are valid probabilities
#         targets   = torch.clamp(targets, min=0.0)
#         targets   = targets / (targets.sum(dim=1, keepdim=True) + 1e-8)
#         # cross‑entropy
#         loss_vec  = -torch.sum(targets * log_probs, dim=1)
#         return loss_vec.mean()


class TorchNNClassifier(BaseEstimator, ClassifierMixin):
    def __init__(
        self,
        hidden_layers=(128,64,32),
        activation='relu',
        lr=1e-2,
        batch_size=64,
        epochs=50,
        temperature=1.0,
        device=None,
        random_state=42
    ):
        self.hidden_layers = hidden_layers
        self.activation    = activation
        self.lr            = lr
        self.batch_size    = batch_size
        self.epochs        = epochs
        self.temperature   = temperature
        self.device        = device
        self.random_state  = random_state

    def _build_model(self, n_features, n_classes):
        layers = []
        in_dim = n_features
        act_fn = nn.ReLU if self.activation=='relu' else nn.Tanh
        for h in self.hidden_layers:
            layers += [nn.Linear(in_dim,h), act_fn()]
            in_dim = h
        layers.append(nn.Linear(in_dim, n_classes))
        return nn.Sequential(*layers)

    def fit(self, X, y, soft_labels = False):
        """
        Dispatch to _fit_hard or _fit_soft based on y’s shape/dtype.
        """
        X = check_array(X)
        # Convert pandas Series/DataFrame to numpy
        if isinstance(y, pd.DataFrame) or isinstance(y, pd.Series):
            y_arr = y.values
        else:
            y_arr = np.asarray(y)

        # Detect soft targets: 2D and rows sum ~1
        if soft_labels:
            return self._fit_soft(X, y_arr)
        else:
            return self._fit_hard(X, y_arr)

    def _fit_hard(self, X, y):
        """Fit on integer labels y (shape [N])."""
        n_samples, n_features = X.shape
        torch.manual_seed(self.random_state)
        self.device_ = torch.device(self.device or ('cuda' if torch.cuda.is_available() else 'cpu'))

        # scale
        self.scaler_ = StandardScaler().fit(X)
        X_scaled = self.scaler_.transform(X)
        X_tensor = torch.tensor(X_scaled, dtype=torch.float32, device=self.device_)

        # map classes
        self.classes_ = np.unique(y)
        self.class_to_idx_ = {c:i for i,c in enumerate(self.classes_)}
        y_idx = np.vectorize(self.class_to_idx_.get)(y)
        y_tensor = torch.tensor(y_idx, dtype=torch.long, device=self.device_)

        ds = TensorDataset(X_tensor, y_tensor)
        loader = DataLoader(ds, batch_size=self.batch_size, shuffle=True)

        # build & train
        self.model_ = self._build_model(n_features, len(self.classes_)).to(self.device_)
        optimizer = optim.SGD(self.model_.parameters(), lr=self.lr)

        for epoch in range(self.epochs):
            self.model_.train()
            for xb, yb in loader:
                xb = xb.to(self.device_)
                yb = yb.to(self.device_)

                optimizer.zero_grad()
                logits = self.model_(xb)                    # <- correct forward
                loss   = F.cross_entropy(logits, yb)        # <- correct loss
                loss.backward()
                optimizer.step()

        return self

    def _fit_soft(self, X, y_soft):
        """Fit on soft targets y_soft (shape [N,C])."""
        n_samples, n_features = X.shape
        n_classes = y_soft.shape[1]
        torch.manual_seed(self.random_state)
        self.device_ = torch.device(self.device or ('cuda' if torch.cuda.is_available() else 'cpu'))

        # scale
        self.scaler_ = StandardScaler().fit(X)
        X_scaled = self.scaler_.transform(X)
        X_tensor = torch.tensor(X_scaled, dtype=torch.float32, device=self.device_)
        y_tensor = torch.tensor(y_soft, dtype=torch.float32, device=self.device_)

        ds = TensorDataset(X_tensor, y_tensor)
        loader = DataLoader(ds, batch_size=self.batch_size, shuffle=True)

        # build & train
        self.model_ = self._build_model(n_features, n_classes).to(self.device_)
        criterion = SoftCrossEntropyLoss(self.temperature)
        optimizer = optim.SGD(self.model_.parameters(), lr=self.lr)

        for epoch in range(self.epochs):
            self.model_.train()
            for xb, yb in loader:
                xb = xb.to(self.device_)
                yb = yb.to(self.device_)
                optimizer.zero_grad()

                # 1) forward pass
                logits = self.model_(xb)               # shape (batch_size, n_classes)
                # if torch.isnan(logits).any() or torch.isinf(logits).any():
                #     print(f"⚠️  NaN/Inf in logits at epoch {epoch}")
                # 2) compute loss
                loss   = criterion(logits, yb)
                # print(logits, yb)

                # 3) backward & step
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model_.parameters(), max_norm=1.0)
                optimizer.step()
                
        return self

    def predict(self, X):
        check_is_fitted(self, ['model_','scaler_'])
        X = check_array(X)
        X_scaled = self.scaler_.transform(X)
        X_tensor = torch.tensor(X_scaled, dtype=torch.float32, device=self.device_)
        self.model_.eval()
        with torch.no_grad():
            logits = self.model_(X_tensor)
            preds = logits.argmax(dim=1).cpu().numpy()
        if hasattr(self, 'class_to_idx_'):
            inv = {i:c for c,i in self.class_to_idx_.items()}
            return np.vectorize(inv.get)(preds)
        return preds

    def predict_proba(self, X):
        check_is_fitted(self, ['model_','scaler_'])
        X = check_array(X)
        X_scaled = self.scaler_.transform(X)
        X_tensor = torch.tensor(X_scaled, dtype=torch.float32, device=self.device_)
        self.model_.eval()
        with torch.no_grad():
            scaled = self.model_(X_tensor) / self.temperature
            return F.softmax(scaled, dim=1).cpu().numpy()


In [ ]:
def expand_with_ties_and_prepare(
    df,
    weight_col='weight',
    target_col_name='target',
    for_nn=False,
    temprature=1.0,
    tie_tolerance: float = 0.0   # 0% default
):
    """
    Given a DataFrame with multiple algorithm runtimes,
    expand rows to duplicate ties (within tolerance) and adjust weights,
    then return X, hard y, soft y, and sample weights.

    Returns
    -------
    X_out        : pd.DataFrame
    y_out        : pd.Series         # hard labels (one per expanded row)
    y_soft_out   : np.ndarray        # soft targets (same shape as X_out × n_algos)
    weights_out  : pd.Series
    """
    expanded_rows = []
    hard_labels   = []
    soft_labels   = []
    weights_list  = []

    # precompute feature set & number of algorithms
    n_alg = len(algorithms)

    for _, row in df.iterrows():
        # 1) raw runtimes and threshold
        runtimes = np.array([row[alg] for alg in algorithms], dtype=float)
        min_rt    = runtimes.min()
        thresh    = (1 + tie_tolerance) * min_rt

        # 2) hard‐tie set
        hard_set = [alg for alg, rt in zip(algorithms, runtimes) if rt <= thresh]

        # 3) soft‐target over *all* algos = softmax(1/runtime)
        inv = 1.0 / (runtimes + 1e-8)
        ex  = np.exp(inv/temprature)
        soft = ex / ex.sum()

        # 4) one expanded row per hard‐tie
        for alg in hard_set:
            if 'kcore_list' in row:
                new_row = row[feature_cols + ['kcore_list']].copy()
            elif for_nn:
                new_row = row[feature_cols + ['fastest_time']].copy()
            else:
                new_row = row[feature_cols].copy()

            expanded_rows.append(new_row)
            hard_labels.append(alg)
            soft_labels.append(soft)   # same soft vector for each copy
            weights_list.append(row[weight_col] / len(hard_set))

    expanded_df = pd.DataFrame(expanded_rows)

    X_out       = expanded_df.drop([target_col_name, weight_col], axis=1, errors='ignore')
    y_out       = pd.Series(hard_labels,  name=target_col_name)
    y_soft_out  = np.vstack(soft_labels)          # shape (n_expanded, n_alg)
    weights_out = pd.Series(weights_list, name=weight_col)

    return X_out, y_out, y_soft_out, weights_out


## Data Splitting and Tie Breaking 

First we split the data and then do the tie breaking.

Only use training set to produce multi-label aware training set with duplicate lines (but different labels)


### **Balanced train-test split based on Y**

In [121]:
train_df, test_df = data_splits.stratify_holdout_train_test_split(
    cat_df,
    target_col='target',
    test_size=0.2,
    random_state=40,
    # how='first'
)

Train set shape: (604, 21)
Test set shape:  (156, 21)


In [122]:
X_train, y_train_hard, y_train_soft, weights_out = expand_with_ties_and_prepare(
    train_df,
    weight_col='weight',
    target_col_name='target',
    tie_tolerance=0.50,
    temprature=1
)

from sklearn.preprocessing import LabelEncoder
# Label‐encode the string targets into integers 0…4 ---
le = LabelEncoder()
y_train = le.fit_transform(y_train_hard)     # e.g. ['AltRB','UMKP',…] → [0,3,…]


print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")

Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


In [123]:
# Instantiate and train
clf_nn_strat_y = TorchNNClassifier(
    hidden_layers=(32, 16),
    # hidden_layers=(8, 4, 2),
    activation='relu',
    lr=0.01,
    batch_size=32,
    epochs=1000,
    temperature=1,
)
clf_nn_strat_y.fit(X_train, y_train_soft, soft_labels=True)


TorchNNClassifier(batch_size=32, epochs=1000, hidden_layers=(32, 16),
                  temperature=1)

#### Inference on 20% dataset

In [132]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=test_df, # for sending the values for test set only.
    clf=clf_nn_strat_y,
    le=le,
    for_nn = False, # set False if you don’t want to use fastest algorithm runtime as a feature
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    # drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./NN_results_with_soft_labels/tie_tol/50/split_and_expand/',
    file_suffix= "inference_test_set_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./NN_results_with_soft_labels/tie_tol/50/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 30.41%

Precision by Algorithm:
  UMKP: 23.81% (21 predictions)
  AltRB: 25.00% (16 predictions)
  kPEX: 0.00% (4 predictions)
  Pivot_Br: 9.09% (11 predictions)
  S_Br: 36.46% (96 predictions)

Recall by Algorithm:
  UMKP: 50.00% (10/20 cases)
  AltRB: 43.48% (10/23 cases)
  kPEX: 23.29% (17/73 cases)
  Pivot_Br: 40.00% (18/45 cases)
  S_Br: 72.55% (37/51 cases)

Time performance metrics:
Time Metrics:
  Total Predicted Time = 44830.10
  Total Optimal Time   = 11002.60
  Time Ratio (pred/opt)= 4.075
  Avg Overhead/case ((pred - opt) / opt)    = 11410.47%


Adding prediction probabilities...

Saving full predictions to './NN_results_with_soft_labels/tie_tol/50/split_and_expand/all_predictions_inference_test_set_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 103 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
                     

#### Re-evalute on entire dataset

In [125]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=clf_nn_strat_y,
    le=le,
    for_nn = False, # set False if you don’t want to use fastest algorithm runtime as a feature
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    # drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./NN_results_with_soft_labels/tie_tol/50/split_and_expand/',
    file_suffix= "inference_entire_data_with_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./NN_results_with_soft_labels/tie_tol/50/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 29.34%

Precision by Algorithm:
  UMKP: 16.82% (107 predictions)
  AltRB: 25.35% (71 predictions)
  kPEX: 0.00% (13 predictions)
  Pivot_Br: 6.67% (90 predictions)
  S_Br: 37.79% (479 predictions)

Recall by Algorithm:
  UMKP: 34.00% (34/100 cases)
  AltRB: 31.03% (36/116 cases)
  kPEX: 22.87% (83/363 cases)
  Pivot_Br: 37.61% (85/226 cases)
  S_Br: 73.83% (189/256 cases)

Time performance metrics:
Time Metrics:
  Total Predicted Time = 268750.90
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 4.018
  Avg Overhead/case ((pred - opt) / opt)    = 5693.65%


Adding prediction probabilities...

Saving full predictions to './NN_results_with_soft_labels/tie_tol/50/split_and_expand/all_predictions_inference_entire_data_with_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 537 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
           d

### **Balanced train-test split based on both Y and k**

In [126]:
train_df, test_df = data_splits.stratify_holdout_train_test_split(
    cat_df,
    target_col='target',
    k_value_col='k_value',
    test_size=0.2,
    random_state=40,
    # how='first'
)

Train set shape: (612, 21)
Test set shape:  (148, 21)


In [127]:
X_train, y_train_hard, y_train_soft, weights_out = expand_with_ties_and_prepare(
    cat_df.drop(index=425, errors='ignore'),
    weight_col='weight',
    target_col_name='target',
    tie_tolerance=0.50,
    temprature=0.5
)

from sklearn.preprocessing import LabelEncoder
# Label‐encode the string targets into integers 0…4 ---
le = LabelEncoder()
y_train = le.fit_transform(y_train_hard)     # e.g. ['AltRB','UMKP',…] → [0,3,…]


print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")

Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


In [129]:
# Instantiate and train
clf_nn_strat_y_K = TorchNNClassifier(
    hidden_layers=(128, 64, 32),
    # hidden_layers=(8, 4, 2),
    activation='relu',
    lr=0.01,
    batch_size=32,
    epochs=500,
    temperature=1,
)
clf_nn_strat_y_K.fit(X_train, y_train_soft, soft_labels=True)


TorchNNClassifier(batch_size=32, epochs=500, temperature=1)

#### Inference on 20% dataset

In [130]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=test_df, # for sending the values for test set only.
    clf=clf_nn_strat_y_K,
    le=le,
    for_nn = False, # set False if you don’t want to use fastest algorithm runtime as a feature
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    # drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./NN_results_with_soft_labels/tie_tol/50/split_and_expand/',
    file_suffix= "inference_test_set_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./NN_results_with_soft_labels/tie_tol/50/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 34.46%

Precision by Algorithm:
  UMKP: 0.00% (19 predictions)
  AltRB: 40.00% (10 predictions)
  kPEX: 33.33% (3 predictions)
  Pivot_Br: 4.55% (22 predictions)
  S_Br: 47.87% (94 predictions)

Recall by Algorithm:
  UMKP: 30.00% (6/20 cases)
  AltRB: 39.13% (9/23 cases)
  kPEX: 26.03% (19/73 cases)
  Pivot_Br: 44.44% (20/45 cases)
  S_Br: 88.24% (45/51 cases)

Time performance metrics:
Time Metrics:
  Total Predicted Time = 42558.40
  Total Optimal Time   = 11002.60
  Time Ratio (pred/opt)= 3.868
  Avg Overhead/case ((pred - opt) / opt)    = 11524.00%


Adding prediction probabilities...

Saving full predictions to './NN_results_with_soft_labels/tie_tol/50/split_and_expand/all_predictions_inference_test_set_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 97 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
                      

#### Re-evalute on entire dataset

In [131]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=clf_nn_strat_y_K,
    le=le,
    for_nn = False, # set False if you don’t want to use fastest algorithm runtime as a feature
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    # drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./NN_results_with_soft_labels/tie_tol/50/split_and_expand/',
    file_suffix= "inference_entire_data_with_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./NN_results_with_soft_labels/tie_tol/50/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 32.76%

Precision by Algorithm:
  UMKP: 3.33% (90 predictions)
  AltRB: 50.00% (38 predictions)
  kPEX: 34.78% (23 predictions)
  Pivot_Br: 10.53% (133 predictions)
  S_Br: 43.07% (476 predictions)

Recall by Algorithm:
  UMKP: 23.00% (23/100 cases)
  AltRB: 30.17% (35/116 cases)
  kPEX: 27.82% (101/363 cases)
  Pivot_Br: 45.13% (102/226 cases)
  S_Br: 81.64% (209/256 cases)

Time performance metrics:
Time Metrics:
  Total Predicted Time = 235609.00
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 3.523
  Avg Overhead/case ((pred - opt) / opt)    = 5810.57%


Adding prediction probabilities...

Saving full predictions to './NN_results_with_soft_labels/tie_tol/50/split_and_expand/all_predictions_inference_entire_data_with_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 511 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
       